# Fitting the Keeling curve
**Introduction to Climate Modeling · November 12, 2026**

How do we learn a function from data, and how do we judge its predictions?

We will fit five models to the Mauna Loa CO₂ record: a line through two points, a least-squares line, a quadratic, a quadratic with an annual cycle, and a neural network. Train each model on earlier observations, then test it on later years.

Download the [notebook and data](https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/keeling_curve_ml.zip), unzip the folder, and open `keeling_curve_ml.ipynb` in the course [LEAP Jupyter environment](https://leap.2i2c.cloud/). Keep the `data` folder beside the notebook. With the supplied data the exercise runs offline; the notebook can also retrieve a fixed copy if the data file is missing.

All code is supplied. Run the cells in order and pause at **Guess before running** and **Discuss** prompts. Write brief answers in the marked cells.

This is a model of **concentration as a function of time**. It does not simulate emissions or the carbon cycle. A good fit alone does not establish the physical cause of a change.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Get and inspect the data

The Keeling curve records atmospheric CO₂ at **Mauna Loa**, rather than the global mean. Concentration is a dry-air mole fraction in **parts per million (ppm)**.

We use a fixed copy of the Scripps CO₂ Program's monthly record, through December 2025. Missing observations are removed without filling the gaps. The original source also appends recent daily data; we exclude that tail so the classroom observations are all monthly.

The data are included with the lab. If needed, the next cell downloads the same source version and prepares the local file. [Scripps CO₂ Program](https://scrippsco2.ucsd.edu/) · Source details: `data/provenance.json`.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from io import BytesIO
import hashlib

data_file = Path("data/mauna_loa_monthly_1958-2025.csv")
SOURCE_URL = 'https://raw.githubusercontent.com/sio-co2o2/keelingcurve_notebooks/80bed51456ac5034a3b5796e7fa820d8b5705112/preliminary_data/mlo/mlo_full_record_now_span.csv'
SOURCE_SHA256 = '16a010800d5a83a624f37ca70f8195f2f9f28f3ee9606c5fb980b20a7cd00018'

if not data_file.exists():
    print("Preparing the fixed classroom data file...")
    with urlopen(SOURCE_URL, timeout=45) as response:
        raw_data = response.read()
    if hashlib.sha256(raw_data).hexdigest() != SOURCE_SHA256:
        raise ValueError("Source data do not match the classroom version. Use the supplied data folder.")
    original = pd.read_csv(BytesIO(raw_data), comment='"', skipinitialspace=True)
    monthly = original[["date", "co2"]].dropna()
    monthly = monthly[(monthly.co2 > 0) & (monthly.date >= 1958) & (monthly.date < 2026)]
    monthly = monthly.sort_values("date")
    data_file.parent.mkdir(parents=True, exist_ok=True)
    monthly.to_csv(data_file, index=False, float_format="%.8f")
print("Using:", data_file)

### Inspect and clean

`date` is time in decimal years; `co2` is concentration in ppm. We retain the original seasonal cycle.

In [ ]:
df = pd.read_csv(data_file)
df.head()

In [ ]:
df_co2_clean = df[["date", "co2"]].dropna().copy()
df_co2_clean = df_co2_clean[df_co2_clean["co2"] > 0].sort_values("date")

In [ ]:
T = df_co2_clean['date'].values
C = df_co2_clean['co2'].values

In [ ]:
plt.plot(T, C)
plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()

In [ ]:
print(f"{len(T)} monthly observations, {T.min():.2f}–{T.max():.2f}")

<a id="response-prediction"></a>

**Guess before running the fits:** Which parts of this record could a straight line capture, and which would it miss?

**Answer:** [Your answer here.]

## 2. Keep later years for testing

For prediction through time, keep the order of the observations. We train on observations **before 2017**, then evaluate on **2017–2025**, which the models have not seen during fitting.

The five model forms and the neural-network settings are specified in advance for this comparison. Repeatedly changing a model after inspecting these test years would turn them into development data; a final assessment would then need a separate untouched period.

In [ ]:
SPLIT_YEAR = 2017
is_train = T < SPLIT_YEAR
T_train, C_train = T[is_train], C[is_train]
T_test, C_test = T[~is_train], C[~is_train]
print(f"Training: {len(T_train)} observations, {T_train.min():.2f}–{T_train.max():.2f}")
print(f"Testing:  {len(T_test)} observations, {T_test.min():.2f}–{T_test.max():.2f}")

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')
plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

## 3. Choose a function and fit its parameters

### Model 1: a line through the first and last training observations

$$C(t)=at+b.$$

Here $a$ is the slope in ppm per year and $b$ is the intercept in ppm. This line uses two selected observations; it does not minimize the error over the whole training record.

In [ ]:
C_start = C_train[0]
T_start = T_train[0]

C_end = C_train[-1]
T_end = T_train[-1]

In [ ]:
print(T_start, C_start, T_end, C_end)

### Estimate the two parameters

Using the first and last training observations $(t_s,C_s)$ and $(t_e,C_e)$,

$$a=\frac{C_e-C_s}{t_e-t_s},\qquad b=C_s-at_s.$$

The next cells calculate these values and extend the fitted line into the test years.

In [ ]:
a = (C_start - C_end)/(T_start - T_end)
b = C_start - a * T_start

print(a, b)

In [ ]:
C_pred_train_Line = a*T_train + b
C_pred_test_Line = a*T_test + b

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_Line, linestyle='--', label='Line')
plt.plot(T_test, C_pred_test_Line, linestyle='--', label='Line')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

### Model 2: a least-squares line

The function still has two parameters,

$$C(t)=at+b,$$

but now we use **all training observations** to choose them. The loss is mean squared error:

$$\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^{n}\left[C_i-\widehat C(t_i)\right]^2.$$

MSE has units of ppm². Scikit-learn's `LinearRegression` finds the coefficients that minimize the training sum of squared errors, which gives the same coefficients as minimizing MSE.

In [ ]:
from sklearn import linear_model

In [ ]:
reg = linear_model.LinearRegression()

In [ ]:
reg.fit(T_train.reshape(-1, 1), C_train.reshape(-1, 1))

In [ ]:
reg.coef_, reg.intercept_

In [ ]:
C_pred_train_LR = reg.predict(T_train.reshape(-1, 1))
C_pred_test_LR = reg.predict(T_test.reshape(-1, 1))

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_Line, linestyle='--', label='Line Train')
plt.plot(T_test, C_pred_test_Line, linestyle='--', label='Line Test')

plt.plot(T_train, C_pred_train_LR, linestyle='--', label='Linear regress Train')
plt.plot(T_test, C_pred_test_LR, linestyle='--', label='Linear regress Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

<a id="response-two-lines"></a>

**Discuss:** Both lines have two parameters. Why are their predictions different, and why can a low training error still leave a large test error?

**Answer:** [Your answer here.]

### Model 3: a quadratic

$$C(\tau)=a\tau^2+b\tau+c,\qquad \tau=t-1958.$$

There are three fitted parameters. Centering the year makes the numerical calculation better conditioned. It changes the coefficients, not the family of quadratic functions.

The inputs below contain $\tau$ and $\tau^2$. The regression also fits an intercept.

In [ ]:
T2_train = np.column_stack((T_train - 1958, (T_train - 1958)**2))
T2_test = np.column_stack((T_test - 1958, (T_test - 1958)**2))

In [ ]:
reg2 = linear_model.LinearRegression()

In [ ]:
reg2.fit(T2_train, C_train.reshape(-1, 1))

In [ ]:
C_pred_train_QR = reg2.predict(T2_train)
C_pred_test_QR = reg2.predict(T2_test)

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_LR, linestyle='--', label='Linear regress Train')
plt.plot(T_test, C_pred_test_LR, linestyle='--', label='Linear regress Test')

plt.plot(T_train, C_pred_train_QR, linestyle='--', label='Quadratic regress Train')
plt.plot(T_test, C_pred_test_QR, linestyle='--', label='Quadratic regress Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

### Model 4: a quadratic plus an annual cycle

$$C(\tau)=a\tau^2+b\tau+c\sin(2\pi\tau/P)+d,\qquad P=1\ \mathrm{year}.$$

We supply the one-year period and the sine phase. The fit chooses **four coefficients**, $a,b,c,d$. A model with an adjustable phase could include an additional cosine term, giving five coefficients.

This is an example of using prior knowledge to choose the function's form.

In [ ]:
tau_train = T_train - 1958
tau_test = T_test - 1958
T3_train = np.column_stack((tau_train, tau_train**2, np.sin(2*np.pi*tau_train)))
T3_test = np.column_stack((tau_test, tau_test**2, np.sin(2*np.pi*tau_test)))

In [ ]:
reg3 = linear_model.LinearRegression()

In [ ]:
reg3.fit(T3_train, C_train.reshape(-1, 1))

In [ ]:
C_pred_train_QRsine = reg3.predict(T3_train)
C_pred_test_QRsine = reg3.predict(T3_test)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(121)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QR, linestyle='--', label='Quadratic regress Train')
plt.plot(T_test, C_pred_test_QR, linestyle='--', label='Quadratic regress Test')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

plt.subplot(122)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QR, linestyle='--', label='Quadratic regress Train')
plt.plot(T_test, C_pred_test_QR, linestyle='--', label='Quadratic regress Test')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.xlim((2010, 2026))
plt.ylim((380, 430))
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

### Model 5: a neural network

A multilayer perceptron combines fitted linear transformations with nonlinear activation functions. Schematically, a two-layer example is

$$\widehat C(\mathbf{x})=A_2\,\sigma(A_1\mathbf{x}+b_1)+b_2.$$

Our network below has three hidden layers with 50, 25 and 10 units. Its inputs are time and the same fixed annual sine used above. Thus both this model and Model 4 receive information about the annual cycle.

We scale the inputs and target using **training data only**. The predictions are then returned to ppm. The random seed makes the demonstration repeatable within a given software environment.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor

<a id="response-network"></a>

**Guess before running:** Will the neural network's larger parameter count guarantee better predictions for the later years? Explain your guess.

**Answer:** [Your answer here.]

In [ ]:
# Two input features: time and a fixed annual sine.
X = np.column_stack((tau_train, np.sin(2*np.pi*tau_train)))
X_test = np.column_stack((tau_test, np.sin(2*np.pi*tau_test)))

network = MLPRegressor(hidden_layer_sizes=(50, 25, 10),
                       activation="relu", solver="adam",
                       max_iter=5000, tol=1e-6, random_state=0)
model = TransformedTargetRegressor(
    regressor=make_pipeline(StandardScaler(), network),
    transformer=StandardScaler(),
)
model.fit(X, C_train)

C_pred_train_NN = model.predict(X)
C_pred_test_NN = model.predict(X_test)
mlp = model.regressor_.named_steps["mlpregressor"]

In [ ]:
n_params = sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print(f"Total trainable parameters: {n_params}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(121)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.plot(T_train, C_pred_train_NN, linestyle='--', label='NN Train')
plt.plot(T_test, C_pred_test_NN, linestyle='--', label='NN Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

plt.subplot(122)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.plot(T_train, C_pred_train_NN, linestyle='--', label='NN Train')
plt.plot(T_test, C_pred_test_NN, linestyle='--', label='NN Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.xlim((2010, 2026))
plt.ylim((380, 430))
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

## 4. Compare training and test error

Compare all five models using the same observations and the same loss. The logarithmic vertical axis makes both small and large errors visible.

Also compare the number of fitted parameters. Complexity and prediction skill are different properties.

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
MSE_test = {}
MSE_train = {}
MSE_test['Line'] = mean_squared_error(C_test, C_pred_test_Line)
MSE_test['LR'] = mean_squared_error(C_test, C_pred_test_LR)
MSE_test['QR'] = mean_squared_error(C_test, C_pred_test_QR)
MSE_test['QR_sine'] = mean_squared_error(C_test, C_pred_test_QRsine)
MSE_test['NN'] = mean_squared_error(C_test, C_pred_test_NN)

MSE_train['Line'] = mean_squared_error(C_train, C_pred_train_Line)
MSE_train['LR'] = mean_squared_error(C_train, C_pred_train_LR)
MSE_train['QR'] = mean_squared_error(C_train, C_pred_train_QR)
MSE_train['QR_sine'] = mean_squared_error(C_train, C_pred_train_QRsine)
MSE_train['NN'] = mean_squared_error(C_train, C_pred_train_NN)


In [ ]:
model_names = list(MSE_train)
counts = [2, 2, 3, 4, n_params]
scores = pd.DataFrame({
    "Model": ["Endpoint line", "Least-squares line", "Quadratic", "Quadratic + sine", "Neural network"],
    "Parameters": counts,
    "Training MSE (ppm²)": [MSE_train[name] for name in model_names],
    "Test MSE (ppm²)": [MSE_test[name] for name in model_names],
}).set_index("Model")
display(scores.round(3))

ax = scores[["Training MSE (ppm²)", "Test MSE (ppm²)"]].plot.bar(
    figsize=(9, 4.5), color=["#2374a5", "#c54b26"], rot=20)
ax.set(yscale="log", ylabel="Mean squared error (ppm²)",
       title="Fitting earlier years and predicting later years")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<a id="response-evaluation"></a>

**Discuss:** Which model has the smallest training error? Which has the smallest test error? Use the table to support your answer. Does this comparison explain the physical causes of the CO₂ increase, or justify extrapolating arbitrarily far into the future?

**Answer:** [Your answer here.]

## Sources and reuse

Adapted from Dhruv Balwada's [original Keeling-curve Colab exercise](https://colab.research.google.com/drive/1rcy60oLVwHnzgB952nSGudL5fR_4foOB).

Data: **Scripps CO₂ Program**, Scripps Institution of Oceanography, UC San Diego. The fixed source version, processing choices and checksums are recorded in `data/provenance.json`. The source record is subject to calibration and quality-control revisions.

Suggested source citation: C. D. Keeling and colleagues (2005), *Atmospheric CO₂ and ¹³CO₂ exchange with the terrestrial biosphere and oceans from 1978 to 2000: observations and carbon cycle implications*, in *A History of Atmospheric CO₂ and Its Effects on Plants, Animals, and Ecosystems*, pp. 83–113.

[Scripps CO₂ data](https://scrippsco2.ucsd.edu/data/atmospheric_co2/) · [Scikit-learn: linear regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [Scikit-learn: neural-network regression](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)
